LLM

In [123]:
from langchain_groq.chat_models import ChatGroq
import os
Groq_Token = os.getenv('CHAT_GROQ_API_KEY')

In [124]:
model_name = "llama-3.1-8b-instant"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)
query="What are you?"
answer = llm.invoke(query)
print(answer.content)

I'm an artificial intelligence model known as a large language model (LLM) or a conversational AI. I'm a computer program designed to understand and generate human-like text responses to a wide range of questions and topics. I'm often referred to as a "chatbot" or a "virtual assistant."

I was trained on a massive dataset of text from the internet, books, and other sources, which allows me to learn patterns and relationships in language. This training enables me to generate responses to user input, from simple answers to complex discussions.

I don't have personal experiences, emotions, or consciousness like humans do. I exist solely to provide information, answer questions, and assist with tasks to the best of my abilities based on my training and knowledge.

How can I assist you today?


Tool Creation and Management

In [125]:
from abc import ABC, abstractmethod
import requests
import os

class Tool(ABC):
    name = ""
    description = ""
    @abstractmethod
    def run(self, **kwargs):
        pass

API Tools

Weather Tool

In [126]:
class WeatherTool(Tool):
    name = "weather"
    description = (
        "Get current weather information for a city."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, city):
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": city,
            "appid": self.api_key,
            "units": "metric"
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return f"Error: {response.json().get('message')}"
        data = response.json()
        weather = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        return (
            f"Weather in {city}:\n"
            f"Temperature: {temp}°C\n"
            f"Condition: {weather}\n"
            f"Humidity: {humidity}%"
        )

Currency Exchange Rate Tool

In [127]:
class ExchangeRateTool(Tool):
    name = "exchange_rate"
    description = (
        "Get exchange rate between two currencies."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, base_currency, target_currency):
        url = (
            f"https://v6.exchangerate-api.com/v6/"
            f"{self.api_key}/latest/{base_currency.upper()}"
        )
        response = requests.get(url)
        if response.status_code != 200:
            return "Failed to fetch exchange rates."
        data = response.json()
        if data["result"] != "success":
            return data.get("error-type", "Unknown error")
        rates = data["conversion_rates"]
        if target_currency.upper() not in rates:
            return f"Currency '{target_currency}' not found."
        rate = rates[target_currency.upper()]
        return (
            f"Exchange Rate:\n"
            f"1 {base_currency.upper()} = "
            f"{rate} {target_currency.upper()}"
        )

News Tool

In [128]:
class NewsTool(Tool):
    name = "news"
    description = (
        "Get latest news headlines for a topic."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, topic="technology", limit=5):
        url = "https://newsapi.org/v2/everything"
        params = {
            "q": topic,
            "sortBy": "publishedAt",
            "language": "en",
            "pageSize": limit,
            "apiKey": self.api_key
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return "Failed to fetch news."
        data = response.json()
        if data["status"] != "ok":
            return data.get("message", "Unknown error")
        articles = data["articles"]
        if not articles:
            return f"No news found for '{topic}'."
        result = [f"Latest news about {topic}:\n"]
        for i, article in enumerate(articles, start=1):
            result.append(
                f"{i}. {article['title']}\n"
                f"   Source: {article['source']['name']}\n"
                f"   URL: {article['url']}\n"
            )
        return "\n".join(result)

TimeZone Tool

In [129]:
class TimezoneTool(Tool):
    name = "timezone"
    description = (
        "Get timezone information from a timezone name."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, timezone):
        url = "https://api.api-ninjas.com/v1/timezone"
        headers = {
            "X-Api-Key": self.api_key
        }
        params = {
            "timezone": timezone
        }
        response = requests.get(
            url,
            headers=headers,
            params=params
        )
        if response.status_code != 200:
            return f"Error: {response.text}"
        data = response.json()
        return (
            f"Timezone: {data['timezone']}\n"
            f"Local Time: {data['local_time']}\n"
            f"UTC Offset: {data['utc_offset']} seconds"
        )

Email Tool

In [130]:
class EmailTool(Tool):
    name = "email"
    description = "Send an email using Resend."
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, to, subject, body):
        import requests
        url = "https://api.resend.com/emails"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        # CONVERT TEXT BODY TO NICE HTML
        formatted_body = body.replace("\n", "<br>")
        html_content = f"""
        <html>
        <body style="font-family: Arial, sans-serif; line-height: 1.6;">
            <h2>{subject}</h2>
            <div>
                {formatted_body}
            </div>
        </body>
        </html>
        """
        payload = {
            "from": "onboarding@resend.dev",
            "to": [to],
            "subject": subject,
            "html": html_content
        }
        response = requests.post(
            url,
            headers=headers,
            json=payload
        )
        if response.status_code not in [200, 201]:
            return f"Error: {response.text}"
        data = response.json()
        return (
            "Email sent successfully!\n"
            f"Email ID: {data.get('id')}"
        )


Non API Tools

Holiday Tool

In [131]:
class PublicHolidayTool(Tool):
    name = "public_holiday"
    description = (
        "Check upcoming public holidays for a country using its 2-letter ISO code (e.g. JP, US, IN)."
    )
    def run(self, country_code, year=2026):
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code.upper()}"
        response = requests.get(url)
        if response.status_code != 200:
            return f"Could not fetch holidays for '{country_code}'."
        holidays = response.json()
        if not holidays:
            return f"No public holidays found for {country_code.upper()} in {year}."
        lines = [f"Public Holidays in {country_code.upper()} ({year}):"]
        for h in holidays[:6]:
            lines.append(f"  {h['date']}: {h['localName']} ({h['name']})")
        return "\n".join(lines)

Wikipedia Tool

In [132]:
class WikipediaTool(Tool):
    name = "wikipedia"
    description = (
        "Get a Wikipedia summary for a city, country, or landmark."
    )
    def run(self, query):
        url = (
            "https://en.wikipedia.org/api/rest_v1/page/summary/"
            + query.replace(" ", "_")
        )
        response = requests.get(url, headers={"User-Agent": "TravelAgent/1.0"})
        if response.status_code != 200:
            return f"No Wikipedia article found for '{query}'."
        data = response.json()
        extract = data.get("extract", "No summary available.")
        return f"{data.get('title', query)}\n{extract[:600]}..."

Tool Registry

In [133]:
# Load API key
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
EXCHANGE_API_KEY = os.getenv("EXCHANGERATE_API_KEY")
NEWS_API_KEY = os.getenv("NEWSAPI_API_KEY")
NINJAS_API_KEY = os.getenv("APININJAS_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(WEATHER_API_KEY),
    "exchange_rate": ExchangeRateTool(EXCHANGE_API_KEY),
    "news": NewsTool(NEWS_API_KEY),
    "timezone": TimezoneTool(NINJAS_API_KEY),
    "email": EmailTool(RESEND_API_KEY),
    "public_holiday": PublicHolidayTool(),
    "wikipedia": WikipediaTool()
}

Tool Execution

In [134]:
def execute_tool(tool_name, **kwargs):
    tool = TOOLS.get(tool_name)
    if not tool:
        return f"Error: Tool '{tool_name}' not found. Available: {list(TOOLS.keys())}"
    try:
        return tool.run(**kwargs)
    except TypeError as e:
        return f"Error: Wrong arguments for '{tool_name}': {e}"
    except Exception as e:
        return f"Error executing '{tool_name}': {e}"

Define Signatures for the Prompt

In [135]:
# JSON signatures shown to the LLM so it knows exactly what arguments to pass
TOOL_SIGNATURES = {
    "weather":        '{"city": "<city name>"}',
    "exchange_rate":  '{"base_currency": "<3-letter code>", "target_currency": "<3-letter code>"}',
    "news":           '{"topic": "<topic or city name>", "limit": <1-5>}',
    "timezone":       '{"timezone": "<IANA timezone e.g. Asia/Tokyo>"}',
    "email":          '{"to": "<email address>", "subject": "<subject>", "body": "<full briefing text>"}',
    "public_holiday": '{"country_code": "<2-letter ISO code e.g. JP>", "year": <4-digit year>}',
    "wikipedia":      '{"query": "<city or country name>"}',
}

Results

In [ ]:
result = execute_tool(
    "weather",
    city="Bangalore"
)
print(result)

In [ ]:
result = execute_tool(
    "exchange_rate",
    base_currency="USD",
    target_currency="INR"
)
print(result)

In [ ]:
result = execute_tool(
    "news",
    topic="Stocks",
    limit=3
)
print(result)

In [ ]:
result = execute_tool(
    "timezone",
    timezone="Europe/London"
)

print(result)

In [ ]:
result = execute_tool(
    "email",
    to="ogiboyinaakash@gmail.com",
    subject="Hello",
    body="This email was sent by my AI agent."
)
print(result)

In [ ]:
result = execute_tool(
    "public_holiday",
    country_code="US",
    year=2026
)
print(result)

Public Holidays in JP (2026):
  2026-01-01: 元日 (New Year's Day)
  2026-01-12: 成人の日 (Coming of Age Day)
  2026-02-11: 建国記念の日 (Foundation Day)
  2026-02-23: 天皇誕生日 (The Emperor's Birthday)
  2026-03-20: 春分の日 (Vernal Equinox Day)
  2026-04-29: 昭和の日 (Shōwa Day)


In [ ]:
result = execute_tool(
    "wikipedia",
    query="Indian cuisine"
)
print(result)

Culture of India
Indian culture is the heritage of social norms and technologies that originated in or are associated with the ethno-linguistically diverse nation of India, pertaining to the Indian subcontinent until 1947 and the Republic of India post-1947. The term also applies beyond India to countries and cultures whose histories are strongly connected to India by immigration, colonisation, or influence, particularly in South Asia and Southeast Asia. India's languages, religions, dance, music, architecture, food, and customs differ from place to place within the country....


ReAct Agent — Smart Travel Briefing Agent  
<small>
Given a travel query (e.g. "I'm flying from New York to Tokyo next month"), the agent:  
1. Looks up **weather**, **exchange rate**, **timezone**, **public holidays**, **Wikipedia overview**, and **news** for the destination  
2. Synthesizes a complete briefing  
3. Emails the report to the user  

**ReAct loop:** Thought → Action → Observation → repeat → Final Answer</nsmall>

System Prompt

In [136]:
def build_system_prompt():
    tool_list = "\n".join(
        f"  - {name}: {TOOLS[name].description}\n    Input: {sig}"
        for name, sig in TOOL_SIGNATURES.items()
    )
    return f"""You are a Smart Travel Briefing Agent. Given a travel query, gather all relevant information using the tools below, compose a complete briefing, and email it to the user.

Available tools:
{tool_list}

Follow this EXACT format for every reasoning step:

Thought: <reason about what information you still need>
Action: <tool_name>
Action Input: <valid JSON matching the tool's input signature>

The system will reply with:
Observation: <tool result>

Repeat Thought/Action/Action Input until you have used ALL of these tools:
  weather, exchange_rate, timezone, public_holiday, wikipedia, news

Then compose the email:
- The email "body" must be written entirely by you, synthesizing all the Observations you received.
- Include every section: weather, exchange rate, local time, holidays, destination overview, news, and travel tips.
- The email "subject" should mention the destination from the user's query.
- Set "to" to the email address provided by the user.

After the email tool succeeds, write:

Thought: I have gathered all information and sent the briefing email.
Final Answer: <a complete, well-formatted travel briefing — the same content you emailed>

Rules:
- Action Input MUST be valid JSON with double-quoted keys and string values.
- Use only the exact tool names listed above.
- Never fabricate an Observation — wait for the system to provide it.
- Never repeat a tool you already used successfully.
- For timezone use IANA format: Asia/Tokyo, America/New_York, Europe/London, etc.
- For public_holiday use ISO 2-letter country codes: JP, US, IN, GB, DE, etc.
- You decide what city/country/currency/timezone to use based on the user's query — do not ask.
- In the email body string, escape any double quotes with \\\" and write newlines as \\n — never use literal newlines inside JSON strings.
"""

ReAct Loop — Parser + Agent Runner

In [137]:
import json
import re
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

Paser

In [ ]:
def fix_json(raw):
    """
    Fix two common LLM JSON errors before json.loads:
      1. Literal newlines inside string values  ->  \\n
      2. Unicode curly/smart quotes inside values  ->  escaped straight quotes
    Tracks in_string state so only string contents are touched.
    """
    raw = raw.replace('“', '\\"').replace('”', '\\"')
    raw = raw.replace('‘', "\\'").replace('’', "'")

    result = []
    in_string = False
    i = 0
    while i < len(raw):
        ch = raw[i]
        if ch == '\\' and i + 1 < len(raw):
            result.append(ch)
            result.append(raw[i + 1])
            i += 2
        elif ch == '"':
            in_string = not in_string
            result.append(ch)
            i += 1
        elif in_string and ch == '\n':
            result.append('\\n')
            i += 1
        elif in_string and ch == '\r':
            result.append('\\r')
            i += 1
        else:
            result.append(ch)
            i += 1
    return ''.join(result)


def extract_action_input(raw):
    """
    Parse Action Input JSON robustly.
    Primary  : fix_json() + json.loads().
    Fallback : field-by-field regex when the body contains bare double
               quotes that survive fix_json (e.g. news titles).
    """
    try:
        return json.loads(fix_json(raw))
    except (json.JSONDecodeError, Exception):
        pass

    result = {}
    for field in ["to", "subject", "city", "base_currency", "target_currency",
                  "timezone", "country_code", "query", "topic"]:
        m = re.search(rf'"{field}"\s*:\s*"([^"]*)"', raw)
        if m:
            result[field] = m.group(1)
    for field in ["limit", "year"]:
        m = re.search(rf'"{field}"\s*:\s*(\d+)', raw)
        if m:
            result[field] = int(m.group(1))
    body_m = re.search(r'"body"\s*:\s*"(.*?)"\s*\}?\s*$', raw, re.DOTALL)
    if body_m:
        body = body_m.group(1).replace('\\n', '\n').replace('\\"', '"')
        result["body"] = body

    return result if result else None


def parse_llm_output(text):
    """
    Position-aware parser.

    When the LLM puts Action + Final Answer in the SAME response (step 7
    pattern: Action: email ... Final Answer: ...), we must execute the
    action first — otherwise the email is never sent.

    Rule: if Action appears BEFORE Final Answer in the text → execute action.
          If Final Answer appears first (or no action) → return final.
    """
    final_match  = re.search(r"Final Answer\s*:\s*(.*)", text, re.DOTALL | re.IGNORECASE)
    action_match = re.search(r"Action\s*:\s*(\w+)", text, re.IGNORECASE)
    input_match  = re.search(r"Action Input\s*:\s*(\{.*\})", text, re.DOTALL | re.IGNORECASE)

    final_pos  = final_match.start()  if final_match  else len(text)
    action_pos = action_match.start() if action_match else len(text)

    # Execute action when it precedes Final Answer
    if action_match and input_match and action_pos < final_pos:
        raw_json = input_match.group(1).strip()
        action_input = extract_action_input(raw_json)
        if action_input is None:
            return {"type": "error", "content": "Could not parse Action Input JSON."}
        return {
            "type": "action",
            "tool": action_match.group(1).strip().lower(),
            "input": action_input,
        }

    # Final Answer with no preceding action
    if final_match:
        return {"type": "final", "content": final_match.group(1).strip()}

    return {"type": "error", "content": f"Could not parse output:\n{text[:300]}"}

Agentic Pipeline

In [139]:
import time
from groq import RateLimitError

def run_travel_agent(query, email_to, max_steps=15, verbose=True):
    """
    ReAct Travel Briefing Agent.

    Two fixes vs the previous version:
      1. Uses a growing message history (not rebuilt from state each step),
         so the LLM sees the real conversation and doesn't re-generate
         a full fake chain in one shot.
      2. Passes stop=["Observation:"] to the LLM so it halts right after
         outputting "Action Input: {...}" and waits for a real observation.
    """
    system_prompt = build_system_prompt()
    separator = "=" * 60

    required_tools = {"weather", "exchange_rate", "timezone", "public_holiday", "wikipedia", "news"}
    completed_tools = set()
    email_sent = False

    # Seed the conversation — never rebuild this list, only append to it
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"{query}\n\nSend the briefing email to: {email_to}"),
    ]

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n{separator}\n  STEP {step}\n{separator}")

        # stop=["Observation:"] forces the LLM to pause after Action Input
        # and prevents it from hallucinating its own observations
        for attempt in range(5):
            try:
                response = llm.invoke(messages, stop=["Observation:"])
                break
            except RateLimitError:
                wait = 3 + attempt * 2
                print(f"\n[Rate Limit] Waiting {wait}s...")
                time.sleep(wait)
        else:
            return "Agent failed due to repeated rate limit errors."

        llm_output = response.content.strip()
        if verbose:
            print(llm_output)

        parsed = parse_llm_output(llm_output)

        # ── Final Answer ───────────────────────────────────────
        if parsed["type"] == "final":
            missing = required_tools - completed_tools
            if missing:
                feedback = f"You still need to use these tools: {sorted(missing)}. Do not write Final Answer yet."
                messages.append(AIMessage(content=llm_output))
                messages.append(HumanMessage(content=f"Observation: {feedback}"))
                continue
            if not email_sent:
                feedback = "You must send the email before writing Final Answer."
                messages.append(AIMessage(content=llm_output))
                messages.append(HumanMessage(content=f"Observation: {feedback}"))
                continue
            if verbose:
                print(f"\n{separator}\n  FINAL ANSWER\n{separator}")
                print(parsed["content"])
            return parsed["content"]

        # ── Tool Call ──────────────────────────────────────────
        elif parsed["type"] == "action":
            tool_name = parsed["tool"]
            tool_input = parsed["input"]

            if tool_name not in TOOLS:
                observation = f"Invalid tool '{tool_name}'. Use only: {list(TOOLS.keys())}"

            elif tool_name in completed_tools and tool_name != "email":
                remaining = sorted(required_tools - completed_tools)
                observation = (
                    f"'{tool_name}' already used successfully. "
                    f"Remaining tools needed: {remaining}"
                )

            elif tool_name == "email":
                missing = required_tools - completed_tools
                if missing:
                    observation = f"Cannot send email yet. Still need: {sorted(missing)}"
                elif email_sent:
                    observation = "Email already sent. Write your Final Answer now."
                else:
                    tool_input["to"] = email_to   # always enforce correct recipient
                    observation = execute_tool(tool_name, **tool_input)
                    if not str(observation).lower().startswith("error"):
                        email_sent = True

            else:
                observation = execute_tool(tool_name, **tool_input)
                if not str(observation).lower().startswith("error"):
                    completed_tools.add(tool_name)

            if verbose:
                print(f"\nObservation: {observation}")

            # Append this turn to the real conversation history
            messages.append(AIMessage(content=llm_output))
            messages.append(HumanMessage(content=f"Observation: {observation}"))

        # ── Parse Error ────────────────────────────────────────
        else:
            feedback = (
                f"{parsed['content'][:200]} "
                "Use the exact format: Thought / Action / Action Input."
            )
            if verbose:
                print(f"\n[Parse Error] {feedback}")
            messages.append(AIMessage(content=llm_output))
            messages.append(HumanMessage(content=f"Observation: {feedback}"))

    return "Agent stopped after maximum steps without a Final Answer."

Run the Agent

In [140]:
result = run_travel_agent(
    query="I'm travelling from Bangalore, India to Tokyo, Japan in July 2026.",
    email_to="ogiboyinaakash@gmail.com",
    verbose=True,
)


  STEP 1
Thought: I need to know the current weather in Tokyo, Japan.
Action: weather
Action Input: {"city": "Tokyo"}

Observation: Weather in Tokyo:
Temperature: 17.08°C
Condition: overcast clouds
Humidity: 81%

  STEP 2
Thought: I need to know the exchange rate between Indian Rupee (INR) and Japanese Yen (JPY).
Action: exchange_rate
Action Input: {"base_currency": "INR", "target_currency": "JPY"}

Observation: Exchange Rate:
1 INR = 1.6696 JPY

  STEP 3
Thought: I need to know the local time in Tokyo, Japan.
Action: timezone
Action Input: {"timezone": "Asia/Tokyo"}

Observation: Timezone: Asia/Tokyo
Local Time: 2026-06-05 20:21:20
UTC Offset: 32400 seconds

  STEP 4
Thought: I need to know the upcoming public holidays in Japan.
Action: public_holiday
Action Input: {"country_code": "JP", "year": 2026}

Observation: Public Holidays in JP (2026):
  2026-01-01: 元日 (New Year's Day)
  2026-01-12: 成人の日 (Coming of Age Day)
  2026-02-11: 建国記念の日 (Foundation Day)
  2026-02-23: 天皇誕生日 (The Emper